# 05 - SHAP Computation for Explainable ML

Load the trained models and processed data from Google Drive, compute SHAP values for all 20 targets, and store per-target cache manifests so resume logic only reuses SHAP artefacts when both the test matrix and model file still match.


## Cell 1 - Setup & Imports

In [ ]:
# Mount Google Drive and install the packages used in this notebook.
from google.colab import drive
drive.mount('/content/drive')

!pip install shap xgboost -q

import hashlib
import json
import os
import pickle
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import shap
import xgboost as xgb

BASE_PATH = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
os.chdir(BASE_PATH)

warnings.filterwarnings('ignore')

print(f'Working directory: {os.getcwd()}')
print(f'SHAP version: {shap.__version__}')
print(f'XGBoost version: {xgb.__version__}')


## Cell 2 - Load All Models & Data

In [ ]:
# Canonical feature and target order used by preprocessing, training, and SHAP.
FEATURE_NAMES = [
    'SG 1 MVA',
    'SG 2 MVA',
    'SG 3 MVA',
    'System Loading',
    'CIG MW',
    'Outage SG',
    'SG 1 MW',
    'SG 2 MW',
    'SG 3 MW',
]
TARGET_ORDER = (
    [f'RoCoF Bus {i}' for i in range(1, 10)]
    + ['RoCoF Worst']
    + [f'Nadir Bus {i}' for i in range(1, 10)]
    + ['Nadir Worst']
)


def _filename_to_target(filename):
    base = os.path.splitext(filename)[0]
    return base.replace('_', ' ')


def _sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()


def _sha256_file(path):
    with open(path, 'rb') as file_obj:
        return _sha256_bytes(file_obj.read())


def _manifest_path(per_target_path):
    stem, _ = os.path.splitext(per_target_path)
    return f'{stem}_manifest.json'


def _load_cached_result(per_target_path, target_name, x_test_hash, model_hash):
    manifest_path = _manifest_path(per_target_path)
    if not os.path.exists(per_target_path):
        return None
    if not os.path.exists(manifest_path):
        print(f'  WARNING: manifest missing for {target_name}; recomputing SHAP cache.')
        return None

    with open(manifest_path, 'r', encoding='utf-8') as file_obj:
        manifest = json.load(file_obj)

    manifest_target = manifest.get('target_name', manifest.get('target'))
    hashes_match = (
        manifest.get('x_test_sha256') == x_test_hash
        and manifest.get('model_sha256') == model_hash
        and manifest_target == target_name
    )
    if hashes_match:
        with open(per_target_path, 'rb') as file_obj:
            return pickle.load(file_obj)

    print(f'  WARNING: cache provenance mismatch for {target_name}; recomputing SHAP values.')
    return None


def _write_cached_result(per_target_path, result, target_name, x_test_hash, model_hash):
    with open(per_target_path, 'wb') as file_obj:
        pickle.dump(result, file_obj)

    manifest = {
        'target_name': target_name,
        'x_test_sha256': x_test_hash,
        'model_sha256': model_hash,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    with open(_manifest_path(per_target_path), 'w', encoding='utf-8') as file_obj:
        json.dump(manifest, file_obj, indent=2)


def _load_xgb_model(path_to_ubj, x_reference):
    model = xgb.XGBRegressor()
    model.load_model(path_to_ubj)
    assert callable(getattr(model, 'predict', None)), f'Loaded XGBoost model has no predict(): {path_to_ubj}'
    sample_pred = np.asarray(model.predict(x_reference[:1]))
    assert sample_pred.shape == (1,), (
        f'Loaded XGBoost model predict() failed for {path_to_ubj}: '
        f'got shape {sample_pred.shape}.'
    )
    booster = model.get_booster()
    assert booster is not None and len(booster.get_dump()) > 0, (
        f'Loaded XGBoost model has no valid booster: {path_to_ubj}'
    )
    return model


models_dir = os.path.join(BASE_PATH, 'models', 'MLP')
models = {}
for filename in sorted(os.listdir(models_dir)):
    if filename.endswith('.pkl') and '_predictions' not in filename:
        target_name = _filename_to_target(filename)
        path = os.path.join(models_dir, filename)
        models[target_name] = joblib.load(path)
        print(f'  Loaded MLP: {filename} -> "{target_name}"')

processed_dir = os.path.join(BASE_PATH, 'data', 'processed')
X_train = joblib.load(os.path.join(processed_dir, 'X_train_scaled.pkl'))
X_test = joblib.load(os.path.join(processed_dir, 'X_test_scaled.pkl'))
y_test_dict = joblib.load(os.path.join(processed_dir, 'y_test_dict.pkl'))
scaler = joblib.load(os.path.join(processed_dir, 'scaler_X.pkl'))
y_test = pd.DataFrame(y_test_dict)
print('\nLoaded X_train_scaled.pkl, X_test_scaled.pkl, y_test_dict.pkl, and scaler_X.pkl')

if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=FEATURE_NAMES)
if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test, columns=FEATURE_NAMES)

rocof_cols = [f'RoCoF Bus {i}' for i in range(1, 10)]
nadir_cols = [f'Nadir Bus {i}' for i in range(1, 10)]
if 'RoCoF Worst' not in y_test.columns and all(col in y_test.columns for col in rocof_cols):
    y_test['RoCoF Worst'] = y_test[rocof_cols].min(axis=1)
if 'Nadir Worst' not in y_test.columns and all(col in y_test.columns for col in nadir_cols):
    y_test['Nadir Worst'] = y_test[nadir_cols].min(axis=1)

raw_train_path = os.path.join(processed_dir, 'X_train_raw.pkl')
raw_test_path = os.path.join(processed_dir, 'X_test_raw.pkl')
if os.path.exists(raw_train_path) and os.path.exists(raw_test_path):
    X_train_raw = joblib.load(raw_train_path)
    X_test_raw = joblib.load(raw_test_path)
    print('Loaded X_train_raw.pkl and X_test_raw.pkl')
else:
    X_train_raw = pd.DataFrame(
        scaler.inverse_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    X_test_raw = pd.DataFrame(
        scaler.inverse_transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    joblib.dump(X_train_raw, raw_train_path)
    joblib.dump(X_test_raw, raw_test_path)
    print('Reconstructed and saved X_train_raw.pkl and X_test_raw.pkl')

print('\n--- Shapes ---')
print(f'X_train (scaled): {X_train.shape}')
print(f'X_test (scaled):  {X_test.shape}')
print(f'X_train_raw:      {X_train_raw.shape}')
print(f'X_test_raw:       {X_test_raw.shape}')
print(f'y_test:           {y_test.shape}')
print(f'Targets in y_test: {list(y_test.columns)}')


## Cell 3 - Create Background Dataset

In [ ]:
# Methodology: A random subsample of approximately 500 training rows is used as the background set
N_BACKGROUND = 500
RANDOM_STATE = 42

n_train = len(X_train)
sample_fn = getattr(shap, 'sample', None) or getattr(getattr(shap, 'utils', None), 'sample', None)
try:
    if sample_fn:
        background_df = sample_fn(X_train, N_BACKGROUND, random_state=RANDOM_STATE)
    else:
        raise AttributeError('shap.sample not found')
    background_data = background_df.values
    if hasattr(background_df, 'index'):
        bg_iloc = [X_train.index.get_loc(i) for i in background_df.index]
        background_raw = X_train_raw.iloc[bg_iloc].copy()
        idx = np.array(bg_iloc)
    else:
        idx = np.arange(len(background_data))
        background_raw = X_train_raw.iloc[idx].copy()
except Exception:
    np.random.seed(RANDOM_STATE)
    idx = np.random.choice(n_train, size=min(N_BACKGROUND, n_train), replace=False)
    background_data = X_train.iloc[idx].values
    background_raw = X_train_raw.iloc[idx].copy()

print(f'Background dataset shape: {background_data.shape}')
print(f'Background raw shape:    {background_raw.shape}')
print(f'Indices (iloc) saved for reproducibility.')

## Cell 4 - Compute SHAP Values for All 20 Models

In [ ]:
shap_out_dir = os.path.join(BASE_PATH, 'shap_results')
os.makedirs(shap_out_dir, exist_ok=True)
mlp_shap_dir = os.path.join(shap_out_dir, 'mlp')
os.makedirs(mlp_shap_dir, exist_ok=True)
print(f'Output directory: {shap_out_dir}\n')

X_test_values = X_test.values
x_test_hash = _sha256_bytes(X_test_values.tobytes())
n_test = X_test_values.shape[0]


def _kernel_explainer_result(model, x_values, background):
    explainer = shap.KernelExplainer(model.predict, background)
    shap_values = np.asarray(explainer.shap_values(x_values))
    expected_value = explainer.expected_value
    if isinstance(expected_value, np.ndarray):
        expected_value = float(np.asarray(expected_value).flatten()[0])
    else:
        expected_value = float(expected_value)
    return {'shap_values': shap_values, 'expected_value': expected_value}


def _tree_explainer_result(model, x_values):
    explainer = shap.TreeExplainer(model)
    shap_values = np.asarray(explainer.shap_values(x_values))
    expected_value = float(np.asarray(explainer.expected_value).flatten()[0])
    return {'shap_values': shap_values, 'expected_value': expected_value}


def _load_joblib_models(model_dir):
    loaded_models = {}
    model_paths = {}
    if not os.path.exists(model_dir):
        return loaded_models, model_paths
    for filename in sorted(os.listdir(model_dir)):
        if filename.endswith('.pkl') and '_predictions' not in filename:
            target_name = _filename_to_target(filename)
            path = os.path.join(model_dir, filename)
            loaded_models[target_name] = joblib.load(path)
            model_paths[target_name] = path
    return loaded_models, model_paths


def _load_xgb_models(model_dir, x_reference):
    loaded_models = {}
    model_paths = {}
    if not os.path.exists(model_dir):
        return loaded_models, model_paths
    for filename in sorted(os.listdir(model_dir)):
        if filename.endswith('.ubj'):
            target_name = _filename_to_target(filename)
            path = os.path.join(model_dir, filename)
            loaded_models[target_name] = _load_xgb_model(path, x_reference)
            model_paths[target_name] = path
    return loaded_models, model_paths



# MLP SHAP - KernelExplainer

all_shap_results = {}
total_start = time.time()

for i, target_name in enumerate(TARGET_ORDER, start=1):
    if target_name not in models:
        print(f'[{i}/20] SKIP (model not found): {target_name}')
        continue

    safe_name = target_name.replace(' ', '_')
    model_path = os.path.join(BASE_PATH, 'models', 'MLP', f'{safe_name}.pkl')
    per_target_path = os.path.join(mlp_shap_dir, f'shap_{safe_name}.pkl')
    model_hash = _sha256_file(model_path)
    cached_result = _load_cached_result(per_target_path, target_name, x_test_hash, model_hash)
    if cached_result is not None:
        all_shap_results[target_name] = cached_result
        print(f'[{i}/20] SKIP (manifest match): {target_name}')
        continue

    model = models[target_name]
    start_time = time.time()
    result = _kernel_explainer_result(model, X_test_values, background_data)
    elapsed = time.time() - start_time
    all_shap_results[target_name] = result
    _write_cached_result(per_target_path, result, target_name, x_test_hash, model_hash)
    print(
        f'[{i}/20] {target_name} | EV={result["expected_value"]:.4f} | '
        f'time={elapsed:.1f}s | shape={result["shap_values"].shape} | SAVED'
    )

total_elapsed = time.time() - total_start
print(f'\n--- MLP SHAP time: {total_elapsed/60:.1f} min --- {len(all_shap_results)}/20 targets')

# RF SHAP - TreeExplainer

rf_models_dir = os.path.join(BASE_PATH, 'models', 'RF')
rf_models, rf_model_paths = _load_joblib_models(rf_models_dir)
rf_shap_dir = os.path.join(shap_out_dir, 'rf')
os.makedirs(rf_shap_dir, exist_ok=True)
all_rf_shap = {}

if rf_models:
    print('\n=== RF SHAP (TreeExplainer) ===')
    for i, target_name in enumerate(TARGET_ORDER, start=1):
        if target_name not in rf_models:
            continue
        safe_name = target_name.replace(' ', '_')
        per_target_path = os.path.join(rf_shap_dir, f'shap_{safe_name}.pkl')
        model_hash = _sha256_file(rf_model_paths[target_name])
        cached_result = _load_cached_result(per_target_path, target_name, x_test_hash, model_hash)
        if cached_result is not None:
            all_rf_shap[target_name] = cached_result
            print(f'[{i}/20] SKIP (manifest match): {target_name}')
            continue

        start_time = time.time()
        result = _tree_explainer_result(rf_models[target_name], X_test_values)
        elapsed = time.time() - start_time
        all_rf_shap[target_name] = result
        _write_cached_result(per_target_path, result, target_name, x_test_hash, model_hash)
        print(f'[{i}/20] {target_name} | EV={result["expected_value"]:.4f} | time={elapsed:.1f}s | SAVED')

    with open(os.path.join(rf_shap_dir, 'all_shap_values_rf.pkl'), 'wb') as file_obj:
        pickle.dump(all_rf_shap, file_obj)
    print(f'RF SHAP complete: {len(all_rf_shap)} targets')
else:
    print('\nRF models dir not found - skipping RF SHAP')


# XGBoost SHAP - TreeExplainer from native .ubj files

xgb_models_dir = os.path.join(BASE_PATH, 'models', 'XGBoost')
xgb_models, xgb_model_paths = _load_xgb_models(xgb_models_dir, X_test_values)
xgb_shap_dir = os.path.join(shap_out_dir, 'xgboost')
os.makedirs(xgb_shap_dir, exist_ok=True)
all_xgb_shap = {}

if xgb_models:
    print('\n=== XGBoost SHAP (TreeExplainer) ===')
    for i, target_name in enumerate(TARGET_ORDER, start=1):
        if target_name not in xgb_models:
            continue
        safe_name = target_name.replace(' ', '_')
        per_target_path = os.path.join(xgb_shap_dir, f'shap_{safe_name}.pkl')
        model_hash = _sha256_file(xgb_model_paths[target_name])
        cached_result = _load_cached_result(per_target_path, target_name, x_test_hash, model_hash)
        if cached_result is not None:
            all_xgb_shap[target_name] = cached_result
            print(f'[{i}/20] SKIP (manifest match): {target_name}')
            continue

        start_time = time.time()
        result = _tree_explainer_result(xgb_models[target_name], X_test_values)
        elapsed = time.time() - start_time
        all_xgb_shap[target_name] = result
        _write_cached_result(per_target_path, result, target_name, x_test_hash, model_hash)
        print(f'[{i}/20] {target_name} | EV={result["expected_value"]:.4f} | time={elapsed:.1f}s | SAVED')

    with open(os.path.join(xgb_shap_dir, 'all_shap_values_xgb.pkl'), 'wb') as file_obj:
        pickle.dump(all_xgb_shap, file_obj)
    print(f'XGBoost SHAP complete: {len(all_xgb_shap)} targets')
else:
    print('\nXGBoost models dir not found - skipping XGBoost SHAP')


# Shape validation
print('\n=== Shape Validation ===')
for target_name, result in all_shap_results.items():
    assert result['shap_values'].shape == (n_test, 9), (
        f'MLP SHAP shape mismatch for {target_name}: '
        f'got {result["shap_values"].shape}, expected ({n_test}, 9)'
    )
print(f'MLP: All {len(all_shap_results)} targets have correct shape ({n_test}, 9)')

if all_rf_shap:
    for target_name, result in all_rf_shap.items():
        assert result['shap_values'].shape == (n_test, 9), (
            f'RF SHAP shape mismatch for {target_name}: '
            f'got {result["shap_values"].shape}, expected ({n_test}, 9)'
        )
    print(f'RF: All {len(all_rf_shap)} targets have correct shape ({n_test}, 9)')

if all_xgb_shap:
    for target_name, result in all_xgb_shap.items():
        assert result['shap_values'].shape == (n_test, 9), (
            f'XGB SHAP shape mismatch for {target_name}: '
            f'got {result["shap_values"].shape}, expected ({n_test}, 9)'
        )
    print(f'XGBoost: All {len(all_xgb_shap)} targets have correct shape ({n_test}, 9)')


## Cell 5 - Save All SHAP Results

In [ ]:
with open(os.path.join(mlp_shap_dir, 'all_shap_values_mlp.pkl'), 'wb') as file_obj:
    pickle.dump(all_shap_results, file_obj)
with open(os.path.join(shap_out_dir, 'all_shap_values.pkl'), 'wb') as file_obj:
    pickle.dump(all_shap_results, file_obj)
print('Saved: shap_results/mlp/all_shap_values_mlp.pkl and shap_results/all_shap_values.pkl')

joblib.dump(
    {'indices': idx, 'n_background': N_BACKGROUND, 'random_state': RANDOM_STATE},
    os.path.join(shap_out_dir, 'background_indices.pkl'),
)
joblib.dump(background_raw, os.path.join(shap_out_dir, 'background_raw.pkl'))
print('Saved: shap_results/background_indices.pkl and shap_results/background_raw.pkl')

joblib.dump(X_test_raw, os.path.join(shap_out_dir, 'X_test_raw.pkl'))
print('Saved: shap_results/X_test_raw.pkl')

print('\nAll SHAP results, cache manifests, and plotting artefacts saved.')


## Cell 6 - Quick Validation (Additivity Check)

In [ ]:
# Additivity must fail loudly if any cached or newly computed SHAP values drift.
np.random.seed(42)
n_check = 5
sample_idx = np.random.choice(n_test, size=min(n_check, n_test), replace=False)


def _assert_additivity(algorithm_name, target_name, result, model, x_values, atol, rtol=0.0):
    expected_value = float(result['expected_value'])
    shap_values = np.asarray(result['shap_values'])
    for iloc_idx in sample_idx:
        pred = float(np.asarray(model.predict(x_values[iloc_idx:iloc_idx + 1])).ravel()[0])
        recon = float(expected_value + shap_values[iloc_idx].sum())
        if not np.isclose(pred, recon, rtol=rtol, atol=atol):
            raise AssertionError(
                f'{algorithm_name} additivity failed for {target_name} at iloc {iloc_idx}: '
                f'pred={pred:.6f}, recon={recon:.6f}, diff={pred - recon:.6f}'
            )


print('=== MLP Additivity (KernelExplainer, atol=1e-2, rtol=1e-2) ===')
rows = []
for target_name in TARGET_ORDER:
    if target_name not in all_shap_results:
        continue
    _assert_additivity(
        'MLP',
        target_name,
        all_shap_results[target_name],
        models[target_name],
        X_test_values,
        atol=1e-2,
        rtol=1e-2,
    )
    shap_values = np.asarray(all_shap_results[target_name]['shap_values'])
    rows.append({
        'Target': target_name,
        'Expected value': f"{all_shap_results[target_name]['expected_value']:.4f}",
        'Mean |SHAP| per feature': f'{np.abs(shap_values).mean(axis=0).mean():.4f}',
        'Validation': 'Pass',
    })
summary = pd.DataFrame(rows)
display(summary)

for algorithm_name, algorithm_shap, algorithm_models, atol in [
    ('RF', all_rf_shap, rf_models, 1e-4),
    ('XGBoost', all_xgb_shap, xgb_models, 1e-4),
]:
    if not algorithm_shap or not algorithm_models:
        continue
    print(f'\n=== {algorithm_name} Additivity (TreeExplainer, atol={atol}) ===')
    for target_name in TARGET_ORDER:
        if target_name not in algorithm_shap or target_name not in algorithm_models:
            continue
        _assert_additivity(
            algorithm_name,
            target_name,
            algorithm_shap[target_name],
            algorithm_models[target_name],
            X_test_values,
            atol=atol,
        )
        print(f'  {target_name}: PASS')
